## Linear Classification

In image classification, the goal is to assign a label to an input image from a fixed set of categories.

Instead of comparing a test image with all training images (as in kNN), we define a **parametric model** that maps input data to class scores.

This approach consists of two components:
- A **score function** that maps input data to class scores
- A **loss function** that measures how well the predicted scores match the true labels

---

## Linear Score Function

We define a linear mapping from input data to class scores:

$$
f(x_i, W, b) = W x_i + b
$$

where:
- $x_i \in \mathbb{R}^D$ is the input image (flattened into a vector)
- $W \in \mathbb{R}^{K \times D}$ is the weight matrix
- $b \in \mathbb{R}^K$ is the bias vector
- The output is a vector of $K$ class scores

Each row of $W$ corresponds to a separate linear classifier for a class.

---

## Interpretation

The model computes $K$ scores, one for each class. The predicted class is typically the one with the highest score:

$$
\hat{y} = \arg\max_j s_j
$$

The goal is to learn $W$ and $b$ such that the correct class receives a higher score than the incorrect classes.

---

## Advantages

- No need to store training data after training
- Fast prediction (single matrix multiplication)
- Forms the basis for more complex models like neural networks

In [1]:
import numpy as np

def linear_classifier(x, W, b):
    """
    Computes class scores using a linear model.

    Parameters:
    x : numpy array of shape (D,)
        Input feature vector (flattened image)
    W : numpy array of shape (K, D)
        Weight matrix (one row per class)
    b : numpy array of shape (K,)
        Bias vector

    Returns:
    scores : numpy array of shape (K,)
        Score for each class
    """
    # Compute the linear scores
    scores = W @ x + b
    
    return scores


# Example usage
D = 4   # number of input features
K = 3   # number of classes

np.random.seed(0)

# Random input vector (simulating a flattened image)
x = np.random.randn(D)

# Random weight matrix
W = np.random.randn(K, D)

# Random bias vector
b = np.random.randn(K)

# Compute scores
scores = linear_classifier(x, W, b)

# Output results
print("Input:", x)
print("Scores:", scores)
print("Predicted class:", np.argmax(scores))

Input: [1.76405235 0.40015721 0.97873798 2.2408932 ]
Scores: [4.98819662 3.17691475 2.88642156]
Predicted class: 0


## Interpreting a Linear Classifier

A linear classifier computes class scores as a weighted sum of pixel values:

$$
s_j = w_j^\top x + b_j
$$

Each class has its own weight vector $w_j$, which assigns importance to each pixel.

---

## Interpretation of Weights

The weights can be interpreted as templates:

- Positive weights → increase the score when the pixel is present
- Negative weights → decrease the score

For example, a "ship" classifier may assign high positive weights to blue pixels (water) and negative weights to red/green pixels.

---

## Geometric Interpretation

Each image is a point in a high-dimensional space (e.g., $3072$-D for CIFAR-10).

Each class defines a linear function:

$$
w_j^\top x + b_j = 0
$$

This corresponds to a hyperplane that separates the space into regions of positive and negative scores.

- $w$ controls the orientation (rotation)
- $b$ controls the position (translation)

---

## Template Matching View

Each row of $W$ can be seen as a template for a class.

The score is computed as a dot product between the image and the template:

$$
s_j = w_j^\top x
$$

The classifier selects the class whose template best matches the input.

---

## Limitations

A linear classifier can only learn one template per class.

This can lead to issues when a class has multiple variations (e.g., cars of different colors or orientations).

---

## Bias Trick

We can incorporate the bias into the weight matrix by extending the input vector:

$$
x' = [x, 1]
$$

Then the score function becomes:

$$
f(x) = W x'
$$

---

## Data Preprocessing

It is common to normalize input data:

- Subtract the mean (zero-centering)
- Scale values to a fixed range (e.g., $[-1, 1]$)

This improves numerical stability and optimization.

---


## Loss Function

In the previous section, we defined a function that maps input data to class scores, parameterized by weights $W$. The data $(x_i, y_i)$ is fixed, but we can adjust $W$ to make predictions consistent with the ground truth labels.

To measure how well the model performs, we define a **loss function**, which quantifies how "bad" the predictions are.

- High loss → poor predictions  
- Low loss → good predictions  

---

## Multiclass Support Vector Machine (SVM) Loss

The SVM loss enforces that the correct class score should be higher than incorrect class scores by a margin $\Delta$.

For a single example, the loss is defined as:

$$
L_i = \sum_{j \ne y_i} \max(0, \; s_j - s_{y_i} + \Delta)
$$

where:
- $s_j$ is the score for class $j$
- $s_{y_i}$ is the score for the correct class
- $\Delta$ is a hyperparameter (margin)

---

## Example

Suppose:

$$
s = [13, -7, 11], \quad y_i = 0, \quad \Delta = 10
$$

Then:

$$
L_i = \max(0, -7 - 13 + 10) + \max(0, 11 - 13 + 10)
$$

$$
= \max(0, -10) + \max(0, 8)
$$

$$
= 0 + 8 = 8
$$

### Interpretation

- First term → no loss (margin satisfied)
- Second term → margin violated → penalty = 8

The model wants:

$$
s_{y_i} \ge s_j + \Delta
$$

If this is not satisfied, loss is accumulated.

---

## Linear Form of the Loss

Since:

$$
f(x_i; W) = W x_i
$$

we can rewrite the loss as:

$$
L_i = \sum_{j \ne y_i} \max(0, \; w_j^\top x_i - w_{y_i}^\top x_i + \Delta)
$$

---

## Hinge Loss

The function:

$$
\max(0, \cdot)
$$

is called the **hinge loss**.

A variant is the squared hinge loss:

$$
\max(0, \cdot)^2
$$

which penalizes violations more strongly.

---

## Full Loss Over Dataset

We compute the average loss over all examples:

$$
L = \frac{1}{N} \sum_i L_i
$$

---

## Regularization

Problem: Many weight matrices $W$ can give zero loss.

Example:

If $W$ works, then:

$$
\lambda W \quad (\lambda > 1)
$$

also works → ambiguity.

---

## Solution: Regularization

We add a penalty term:

$$
R(W) = \sum_k \sum_l W_{k,l}^2
$$

This is the **L2 regularization**.

---

## Full SVM Loss

$$
L = \frac{1}{N} \sum_i L_i + \lambda R(W)
$$

Expanded form:

$$
L = \frac{1}{N} \sum_i \sum_{j \ne y_i} \max(0, f(x_i;W)_j - f(x_i;W)_{y_i} + \Delta) + \lambda \sum_k \sum_l W_{k,l}^2
$$

---

## Why Regularization?

Regularization discourages large weights.

Example:

$$
w_1 = [1,0,0,0], \quad w_2 = [0.25,0.25,0.25,0.25]
$$

For input:

$$
x = [1,1,1,1]
$$

$$
w_1^\top x = w_2^\top x = 1
$$

But:

$$
\|w_1\|^2 = 1, \quad \|w_2\|^2 = 0.25
$$

👉 $w_2$ is preferred (smaller, more distributed weights)

---

## Effect of Regularization

- Prevents overfitting  
- Encourages smoother models  
- Improves generalization  

---

## Important Notes

- Usually only $W$ is regularized, not $b$
- Loss cannot be exactly zero due to regularization
- $\lambda$ is a hyperparameter (chosen via cross-validation)

---

## Summary

- SVM loss measures how well predictions match labels  
- It enforces a margin between correct and incorrect classes  
- Regularization ensures stable and generalizable solutions  

Minimizing this loss gives us the optimal weights.

In [5]:
def L_i(x, y, W):
  """
  unvectorized version. Compute the multiclass svm loss for a single example (x,y)
  - x is a column vector representing an image (e.g. 3073 x 1 in CIFAR-10)
    with an appended bias dimension in the 3073-rd position (i.e. bias trick)
  - y is an integer giving index of correct class (e.g. between 0 and 9 in CIFAR-10)
  - W is the weight matrix (e.g. 10 x 3073 in CIFAR-10)
  """
  delta = 1.0 # see notes about delta later in this section
  scores = W.dot(x) # scores becomes of size 10 x 1, the scores for each class
  correct_class_score = scores[y]
  D = W.shape[0] # number of classes, e.g. 10
  loss_i = 0.0
  for j in range(D): # iterate over all wrong classes
    if j == y:
      # skip for the true class to only loop over incorrect classes
      continue
    # accumulate loss for the i-th example
    loss_i += max(0, scores[j] - correct_class_score + delta)
  return loss_i

def L_i_vectorized(x, y, W):
  """
  A faster half-vectorized implementation. half-vectorized
  refers to the fact that for a single example the implementation contains
  no for loops, but there is still one loop over the examples (outside this function)
  """
  delta = 1.0
  scores = W.dot(x)
  # compute the margins for all classes in one vector operation
  margins = np.maximum(0, scores - scores[y] + delta)
  # on y-th position scores[y] - scores[y] canceled and gave delta. We want
  # to ignore the y-th position and only consider margin on max wrong class
  margins[y] = 0
  loss_i = np.sum(margins)
  return loss_i

def L(X, y, W):
    """
    Fully-vectorized implementation of the multiclass SVM loss.

    - X holds all training examples as columns.
      Shape: D x N
      Example for CIFAR-10: 3073 x 50000

    - y contains the correct class index for each example.
      Shape: N

    - W contains the class weights.
      Shape: C x D
      Example for CIFAR-10: 10 x 3073
    """

    delta = 1.0

    # Compute scores for all classes and all examples at once
    # scores shape: C x N
    scores = W.dot(X)

    # Select the correct class score for each example
    # correct_scores shape: N
    correct_scores = scores[y, np.arange(X.shape[1])]

    # Compute margins for all classes and all examples
    # margins shape: C x N
    margins = np.maximum(0, scores - correct_scores + delta)

    # Do not count the correct class in the loss
    margins[y, np.arange(X.shape[1])] = 0

    # Sum all margins and average over the number of examples
    loss = np.sum(margins) / X.shape[1]

    return loss

## What Does the SVM Loss Compute?

The SVM loss measures how well the model separates the correct class from the incorrect classes.

For one training example, the loss is:

$$
L_i = \sum_{j \ne y_i} \max(0, s_j - s_{y_i} + \Delta)
$$

A low loss means that the correct class score is higher than incorrect class scores by at least the margin.

A high loss means that one or more incorrect classes are too close to, or higher than, the correct class.

---

## Naive vs Vectorized Implementation

Both implementations compute the same mathematical quantity.

The naive version uses an explicit loop over classes:

$$
L_i = \sum_{j \ne y_i} \max(0, s_j - s_{y_i} + \Delta)
$$

The vectorized version computes all margins at once using NumPy operations.

The result should be the same, but the vectorized version is usually faster because NumPy uses optimized low-level operations.

---

## Runtime Comparison

To compare runtime, we can use Python's `time` module.

We run each function many times and measure how long it takes.

If the vectorized implementation is correct, it should produce the same loss as the naive implementation, but usually in less time.

In [7]:

import time


def L_i(x, y, W):
    """
    Compute the multiclass SVM loss for a single example using loops.
    """
    delta = 1.0
    scores = W.dot(x)
    correct_class_score = scores[y]
    num_classes = W.shape[0]

    loss_i = 0.0

    for j in range(num_classes):
        if j == y:
            continue

        loss_i += max(0, scores[j] - correct_class_score + delta)

    return loss_i


def L_i_vectorized(x, y, W):
    """
    Compute the multiclass SVM loss for a single example using vectorized operations.
    """
    delta = 1.0
    scores = W.dot(x)

    margins = np.maximum(0, scores - scores[y] + delta)
    margins[y] = 0

    loss_i = np.sum(margins)

    return loss_i


def L(X, y, W):
    """
    Compute the multiclass SVM loss for the full dataset using vectorized operations.
    """
    delta = 1.0
    num_examples = X.shape[1]

    scores = W.dot(X)

    correct_scores = scores[y, np.arange(num_examples)]

    margins = np.maximum(0, scores - correct_scores + delta)

    margins[y, np.arange(num_examples)] = 0

    loss = np.sum(margins) / num_examples

    return loss


# Create example data
np.random.seed(42)

D = 3073       # number of features including bias dimension
C = 10         # number of classes
N = 5000       # number of examples

X = np.random.randn(D, N)
W = np.random.randn(C, D)
y = np.random.randint(C, size=N)

x_single = X[:, 0]
y_single = y[0]


# Compare single-example results
loss_naive = L_i(x_single, y_single, W)
loss_vectorized = L_i_vectorized(x_single, y_single, W)

print("Naive single-example loss:", loss_naive)
print("Vectorized single-example loss:", loss_vectorized)
print("Are they equal?", np.allclose(loss_naive, loss_vectorized))


# Measure runtime for naive single-example loss
start_time = time.time()

for _ in range(10000):
    L_i(x_single, y_single, W)

naive_time = time.time() - start_time


# Measure runtime for vectorized single-example loss
start_time = time.time()

for _ in range(10000):
    L_i_vectorized(x_single, y_single, W)

vectorized_time = time.time() - start_time


print("\nRuntime comparison for one example repeated 10,000 times")
print("Naive time:", naive_time)
print("Vectorized time:", vectorized_time)
print("Speedup:", naive_time / vectorized_time)


# Compute full dataset loss
full_loss = L(X, y, W)

print("\nFull dataset loss:", full_loss)

Naive single-example loss: 701.4048534619735
Vectorized single-example loss: 701.4048534619737
Are they equal? True

Runtime comparison for one example repeated 10,000 times
Naive time: 0.2530531883239746
Vectorized time: 0.2655653953552246
Speedup: 0.9528846481880161

Full dataset loss: 291.6123113733923


## Why is the Vectorized Version Slower for Small Data?

Although vectorized implementations are generally faster, they can be slower for small datasets.

### Reason

Vectorized operations (NumPy) have some overhead:
- Memory allocation
- Broadcasting operations
- Internal function calls

For small inputs, this overhead can be larger than the cost of a simple Python loop.

---

### Key Insight

- **Small data** → loop may be faster  
- **Large data** → vectorization is much faster  

---

### Conclusion

Vectorization is preferred in practice because machine learning problems typically involve large datasets, where it provides significant speedups.

## Softmax Classifier and Practical Considerations

### Margin Parameter

The margin $\Delta$ in SVM loss does not need tuning and is typically set to:

$$
\Delta = 1
$$

This is because scaling the weights $W$ changes the score differences, making $\Delta$ less important.

---

### Softmax Classifier

The score function remains:

$$
f(x) = W x
$$

The softmax function converts scores into probabilities:

$$
P(y_i | x_i) = \frac{e^{f_{y_i}}}{\sum_j e^{f_j}}
$$

---

### Cross-Entropy Loss

The loss is defined as:

$$
L_i = -\log \left( \frac{e^{f_{y_i}}}{\sum_j e^{f_j}} \right)
$$

or equivalently:

$$
L_i = -f_{y_i} + \log \sum_j e^{f_j}
$$

---

### Interpretation

- High probability for correct class → low loss  
- Low probability → high loss  

---

### Numerical Stability

To avoid overflow:

$$
f := f - \max_j f_j
$$

This keeps values stable without changing the result.

---

### Key Difference

- SVM → margin-based  
- Softmax → probability-based

In [9]:

def softmax_loss(x, y, W):
    """
    Compute softmax loss for a single example.

    Parameters:
    x : (D,) input vector
    y : int correct class
    W : (C, D) weights

    Returns:
    loss : float
    """
    scores = W.dot(x)

    # Numeric stability trick
    scores -= np.max(scores)

    exp_scores = np.exp(scores)
    probs = exp_scores / np.sum(exp_scores)

    loss = -np.log(probs[y])

    return loss


def softmax_loss_full(X, y, W):
    """
    Fully vectorized softmax loss.
    """
    N = X.shape[1]

    scores = W.dot(X)

    # Numeric stability
    scores -= np.max(scores, axis=0, keepdims=True)

    exp_scores = np.exp(scores)
    probs = exp_scores / np.sum(exp_scores, axis=0, keepdims=True)

    correct_log_probs = -np.log(probs[y, np.arange(N)])

    loss = np.sum(correct_log_probs) / N

    return loss
np.random.seed(0)

D = 5
C = 3
N = 4

X = np.random.randn(D, N)
W = np.random.randn(C, D)
y = np.array([0, 1, 2, 1])

print("Softmax loss:", softmax_loss_full(X, y, W))

Softmax loss: 1.662512180453696


## SVM vs Softmax Classifier

Both SVM and Softmax use the same score function:

$$
f(x) = W x
$$

The difference lies in how these scores are interpreted.

---

## SVM Interpretation

SVM treats the outputs as raw class scores and enforces a margin:

$$
s_{y_i} \ge s_j + \Delta
$$

Once this condition is satisfied, the loss becomes zero.

---

## Softmax Interpretation

Softmax converts scores into probabilities:

$$
P(y_i | x_i) = \frac{e^{f_{y_i}}}{\sum_j e^{f_j}}
$$

The loss is:

$$
L_i = -\log P(y_i | x_i)
$$

---

## Key Difference

- SVM is satisfied once the margin condition is met  
- Softmax always tries to improve probabilities  

---

## Example Comparison

Consider:

$$
[10, -100, -100]
$$

vs

$$
[10, 9, 9]
$$

- SVM treats both equally (margin satisfied)
- Softmax assigns higher loss to the second case

---

## Effect of Regularization

Smaller weights lead to more uniform probabilities:

$$
[1, -2, 0] \rightarrow [0.7, 0.04, 0.26]
$$

$$
[0.5, -1, 0] \rightarrow [0.55, 0.12, 0.33]
$$

---

## Summary

- SVM focuses on margin  
- Softmax focuses on probability distribution  
- Softmax provides interpretable confidence values

In [10]:

def softmax(x):
    x = x - np.max(x)
    exp = np.exp(x)
    return exp / np.sum(exp)

scores1 = np.array([10, -100, -100])
scores2 = np.array([10, 9, 9])

print("Softmax 1:", softmax(scores1))
print("Softmax 2:", softmax(scores2))

Softmax 1: [1.00000000e+00 1.68891188e-48 1.68891188e-48]
Softmax 2: [0.57611688 0.21194156 0.21194156]
